# Data

*\[\[\[model, 123\]\]\]*

In [1]:
!ls yugioh

ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl


In [ ]:
!conda install -c anaconda python=3.8

Solving environment: failed with current_repodata.json, will retry with next repodata source.
Initial quick solve with frozen env failed.  Unfreezing env and trying again.
Solving environment: failed with current_repodata.json, will retry with next repodata source.
Solving environment: failed
Initial quick solve with frozen env failed.  Unfreezing env and trying again.
Solving environment: - 

In [7]:
!pip install flatbuffers=="23.5.26"

Looking in indexes: https://pypi.yandex-team.ru/simple/
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 2015.12.22.1
    Uninstalling flatbuffers-2015.12.22.1:
      Successfully uninstalled flatbuffers-2015.12.22.1


In [1]:
from flatbuffers.compat import import_numpy

In [2]:
import pickle
import numpy as np
import os
import tqdm
# import torch
import gc

import tensorflow 

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2023-09-23 01:45:17.068773: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-09-23 01:45:17.777998: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-09-23 01:45:19.979570: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/shevkunov/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


matrix_approx_zeshel.py


# Open Data loader & context

In [3]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [4]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        #np.random.shuffle(self.requests)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [5]:
# ctx = EvalContextRelevs(load_ment_to_ent_scores("yugioh", shuffle_rows=42), det_attempts=100)

# Games Data loader & context

In [6]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [7]:
# ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA).set_top_games_as_key()

# Models

In [8]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [9]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [10]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [11]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

### open

In [12]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def K_by_min_and_KMeans(X):
    center = X.mean(axis=0)
        
    K = from_labels(
        X,
        KMeans(n_clusters=50, random_state=0).fit(X).labels_  #, n_init="auto").fit(X)
    )


    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [13]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

In [14]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'ApproxNDCGLoss',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True # <<< DIFF HERE
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013


KeyboardInterrupt: 

Exception ignored in: 'sklearn.cluster._k_means_common._relocate_empty_clusters_dense'
Traceback (most recent call last):
  File "<__array_function__ internals>", line 179, in where
KeyboardInterrupt: 

KeyboardInterrupt



In [15]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'ApproxNDCGLoss',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "Winit": "eye",
        "TEinit": "anncur",
        "use_keys_in_train": True # <<< DIFF HERE
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur in


KeyboardInterrupt



In [17]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 20000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": False, # <<< DIFF HERE
        "learning_rate": 0.05
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur in

/home/shevkunov/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:181: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


np.mean(results), mse, len(results) =  0.011876106194690267 inf 2260
slice = train, score = 0.011876106194690267
np.mean(results), mse, len(results) =  0.011885488647581443 inf 1013
slice = test, score = 0.011885488647581443



KeyboardInterrupt



In [18]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 20000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": False, # <<< DIFF HERE
        "learning_rate": 0.01
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur in

KeyboardInterrupt: 

In [20]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": False, # <<< DIFF HERE
        "learning_rate": 0.0005
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur in

np.mean(results), mse, len(results) =  0.598433628318584 1680.5984 2260
slice = train, score = 0.598433628318584
np.mean(results), mse, len(results) =  0.531273445212241 1678.8368 1013
slice = test, score = 0.531273445212241

=== Iteration 21000 ===
np.mean(results), mse, len(results) =  0.5397 1784.6421 100
slice = key, score = 0.5397
np.mean(results), mse, len(results) =  0.5997743362831859 1772.917 2260
slice = train, score = 0.5997743362831859
np.mean(results), mse, len(results) =  0.533938795656466 1769.0977 1013
slice = test, score = 0.533938795656466

=== Iteration 22000 ===
np.mean(results), mse, len(results) =  0.5389 1880.8351 100
slice = key, score = 0.5389
np.mean(results), mse, len(results) =  0.6007522123893806 1877.2738 2260
slice = train, score = 0.6007522123893806
np.mean(results), mse, len(results) =  0.5328726554787758 1878.0199 1013
slice = test, score = 0.5328726554787758

=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.5373 1986.8281 100
slice = k


=== Iteration 44000 ===
np.mean(results), mse, len(results) =  0.5432000000000001 4043.2356 100
slice = key, score = 0.5432000000000001
np.mean(results), mse, len(results) =  0.6152964601769912 4026.1685 2260
slice = train, score = 0.6152964601769912
np.mean(results), mse, len(results) =  0.5359032576505429 4047.0923 1013
slice = test, score = 0.5359032576505429

=== Iteration 45000 ===
np.mean(results), mse, len(results) =  0.542 4170.78 100
slice = key, score = 0.542
np.mean(results), mse, len(results) =  0.6162610619469027 4162.8237 2260
slice = train, score = 0.6162610619469027
np.mean(results), mse, len(results) =  0.5371767028627839 4180.484 1013
slice = test, score = 0.5371767028627839

=== Iteration 46000 ===
np.mean(results), mse, len(results) =  0.5413999999999999 4273.104 100
slice = key, score = 0.5413999999999999
np.mean(results), mse, len(results) =  0.6189867256637168 4254.133 2260
slice = train, score = 0.6189867256637168
np.mean(results), mse, len(results) =  0.536209

np.mean(results), mse, len(results) =  0.6289070796460177 6160.2407 2260
slice = train, score = 0.6289070796460177
np.mean(results), mse, len(results) =  0.5388548864758145 6200.3687 1013
slice = test, score = 0.5388548864758145

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.5359 6276.701 100
slice = key, score = 0.5359
np.mean(results), mse, len(results) =  0.6265530973451328 6288.003 2260
slice = train, score = 0.6265530973451328
np.mean(results), mse, len(results) =  0.5365449160908193 6326.272 1013
slice = test, score = 0.5365449160908193

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.5403 6358.2524 100
slice = key, score = 0.5403
np.mean(results), mse, len(results) =  0.6290265486725664 6370.3 2260
slice = train, score = 0.6290265486725664
np.mean(results), mse, len(results) =  0.5377591312931885 6400.021 1013
slice = test, score = 0.5377591312931885

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.5416000000000001 6417.53 100


=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.5347999999999999 8127.588 100
slice = key, score = 0.5347999999999999
np.mean(results), mse, len(results) =  0.6373584070796461 8143.8564 2260
slice = train, score = 0.6373584070796461
np.mean(results), mse, len(results) =  0.5401480750246791 8191.185 1013
slice = test, score = 0.5401480750246791

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.5343 8309.504 100
slice = key, score = 0.5343
np.mean(results), mse, len(results) =  0.6347035398230089 8302.95 2260
slice = train, score = 0.6347035398230089
np.mean(results), mse, len(results) =  0.5380651530108588 8353.443 1013
slice = test, score = 0.5380651530108588

=== Iteration 93000 ===
np.mean(results), mse, len(results) =  0.5404 8327.073 100
slice = key, score = 0.5404
np.mean(results), mse, len(results) =  0.6352345132743363 8343.317 2260
slice = train, score = 0.6352345132743363
np.mean(results), mse, len(results) =  0.5383020730503455 8393.432 1013


...


=== Iteration 30000 ===
np.mean(results), mse, len(results) =  0.22080000000000002 7416.9604 100
slice = key, score = 0.22080000000000002
np.mean(results), mse, len(results) =  0.2817599110781771 7281.4287 2699
slice = train, score = 0.2817599110781771
np.mean(results), mse, len(results) =  0.24521666666666667 7322.991 1200
slice = test, score = 0.24521666666666667

=== Iteration 31000 ===
np.mean(results), mse, len(results) =  0.21860000000000002 7953.3564 100
slice = key, score = 0.21860000000000002
np.mean(results), mse, len(results) =  0.28458317895516855 7776.23 2699
slice = train, score = 0.28458317895516855
np.mean(results), mse, len(results) =  0.24672500000000006 7826.7783 1200
slice = test, score = 0.24672500000000006

=== Iteration 32000 ===
np.mean(results), mse, len(results) =  0.22149999999999997 8357.391 100
slice = key, score = 0.22149999999999997
np.mean(results), mse, len(results) =  0.28181548721748795 8224.403 2699
slice = train, score = 0.28181548721748795
np.mean

np.mean(results), mse, len(results) =  0.29164875879955543 20065.398 2699
slice = train, score = 0.29164875879955543
np.mean(results), mse, len(results) =  0.24958333333333332 20114.705 1200
slice = test, score = 0.24958333333333332

=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.2266 20866.15 100
slice = key, score = 0.2266
np.mean(results), mse, len(results) =  0.2903112263801408 20905.309 2699
slice = train, score = 0.2903112263801408
np.mean(results), mse, len(results) =  0.24849166666666672 20949.252 1200
slice = test, score = 0.24849166666666672

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.22659999999999997 21470.635 100
slice = key, score = 0.22659999999999997
np.mean(results), mse, len(results) =  0.29183771767321226 21573.58 2699
slice = train, score = 0.29183771767321226
np.mean(results), mse, len(results) =  0.24990833333333332 21594.99 1200
slice = test, score = 0.24990833333333332

=== Iteration 56000 ===
np.mean(results), mse, len(re

np.mean(results), mse, len(results) =  0.2947202667654687 39783.883 2699
slice = train, score = 0.2947202667654687
np.mean(results), mse, len(results) =  0.2489916666666667 39817.09 1200
slice = test, score = 0.2489916666666667

=== Iteration 78000 ===
np.mean(results), mse, len(results) =  0.2289 40268.395 100
slice = key, score = 0.2289
np.mean(results), mse, len(results) =  0.2986254168210448 40636.684 2699
slice = train, score = 0.2986254168210448
np.mean(results), mse, len(results) =  0.2505333333333334 40664.03 1200
slice = test, score = 0.2505333333333334

=== Iteration 79000 ===
np.mean(results), mse, len(results) =  0.23119999999999996 41213.805 100
slice = key, score = 0.23119999999999996
np.mean(results), mse, len(results) =  0.2957021118932938 41549.32 2699
slice = train, score = 0.2957021118932938
np.mean(results), mse, len(results) =  0.24749166666666667 41545.46 1200
slice = test, score = 0.24749166666666667

=== Iteration 80000 ===
np.mean(results), mse, len(results) = 

np.mean(results), mse, len(results) =  0.25034166666666663 62763.33 1200



DATASET =  military
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0700.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0000.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0100.pkl


AssertionError: 

In [21]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "learning_rate": 0.0005
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur in

np.mean(results), mse, len(results) =  0.5955132743362832 1802.1031 2260
slice = train, score = 0.5955132743362832
np.mean(results), mse, len(results) =  0.531253701875617 1799.7714 1013
slice = test, score = 0.531253701875617

=== Iteration 22000 ===
np.mean(results), mse, len(results) =  0.5858 1928.6364 100
slice = key, score = 0.5858
np.mean(results), mse, len(results) =  0.6023849557522123 1918.6433 2260
slice = train, score = 0.6023849557522123
np.mean(results), mse, len(results) =  0.5325074037512341 1921.6398 1013
slice = test, score = 0.5325074037512341

=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.5863 2021.5597 100
slice = key, score = 0.5863
np.mean(results), mse, len(results) =  0.6014778761061946 2012.535 2260
slice = train, score = 0.6014778761061946
np.mean(results), mse, len(results) =  0.534284304047384 2014.825 1013
slice = test, score = 0.534284304047384

=== Iteration 24000 ===
np.mean(results), mse, len(results) =  0.5871 2119.527 100
slice = k


=== Iteration 45000 ===
np.mean(results), mse, len(results) =  0.5994000000000002 4066.5786 100
slice = key, score = 0.5994000000000002
np.mean(results), mse, len(results) =  0.6174424778761062 4038.3374 2260
slice = train, score = 0.6174424778761062
np.mean(results), mse, len(results) =  0.5362783810463968 4071.2786 1013
slice = test, score = 0.5362783810463968

=== Iteration 46000 ===
np.mean(results), mse, len(results) =  0.6038000000000001 4181.268 100
slice = key, score = 0.6038000000000001
np.mean(results), mse, len(results) =  0.6212743362831858 4146.2344 2260
slice = train, score = 0.6212743362831858
np.mean(results), mse, len(results) =  0.5373445212240868 4171.926 1013
slice = test, score = 0.5373445212240868

=== Iteration 47000 ===
np.mean(results), mse, len(results) =  0.6042 4273.632 100
slice = key, score = 0.6042
np.mean(results), mse, len(results) =  0.619079646017699 4233.189 2260
slice = train, score = 0.619079646017699
np.mean(results), mse, len(results) =  0.53836

np.mean(results), mse, len(results) =  0.53928923988154 6082.587 1013
slice = test, score = 0.53928923988154

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.6218 6161.571 100
slice = key, score = 0.6218
np.mean(results), mse, len(results) =  0.6322610619469026 6142.9185 2260
slice = train, score = 0.6322610619469026
np.mean(results), mse, len(results) =  0.5405923000987167 6205.284 1013
slice = test, score = 0.5405923000987167

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.6195 6237.225 100
slice = key, score = 0.6195
np.mean(results), mse, len(results) =  0.628637168141593 6203.5786 2260
slice = train, score = 0.628637168141593
np.mean(results), mse, len(results) =  0.537877591312932 6247.983 1013
slice = test, score = 0.537877591312932

=== Iteration 71000 ===
np.mean(results), mse, len(results) =  0.6181 6312.318 100
slice = key, score = 0.6181
np.mean(results), mse, len(results) =  0.6306061946902655 6300.797 2260
slice = train, score = 0.630606

np.mean(results), mse, len(results) =  0.6372522123893806 7945.881 2260
slice = train, score = 0.6372522123893806
np.mean(results), mse, len(results) =  0.5417176702862784 8021.7773 1013
slice = test, score = 0.5417176702862784

=== Iteration 93000 ===
np.mean(results), mse, len(results) =  0.6283 8033.187 100
slice = key, score = 0.6283
np.mean(results), mse, len(results) =  0.6394557522123894 8039.9277 2260
slice = train, score = 0.6394557522123894
np.mean(results), mse, len(results) =  0.5431095755182627 8117.227 1013
slice = test, score = 0.5431095755182627

=== Iteration 94000 ===
np.mean(results), mse, len(results) =  0.6260000000000001 8176.9834 100
slice = key, score = 0.6260000000000001
np.mean(results), mse, len(results) =  0.6368362831858407 8192.935 2260
slice = train, score = 0.6368362831858407
np.mean(results), mse, len(results) =  0.53928923988154 8285.141 1013
slice = test, score = 0.53928923988154

=== Iteration 95000 ===
np.mean(results), mse, len(results) =  0.6278 8

...

Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1400.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1900.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2100.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0900.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0400.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1000.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0100.pkl
Loaded shape =  (2400, 104520)
Shuffling... (seed = 42)
Best det =  0.0
Current de =  0.0
100 1579 720
self.embed_users['train'].shape =  (1579, 100)
self.embed_games.shape =  (104520, 100)
ANNCur init
self.games2users.shape =  (100, 1


=== Iteration 20000 ===
np.mean(results), mse, len(results) =  0.24729999999999996 6185.9355 100
slice = key, score = 0.24729999999999996
np.mean(results), mse, len(results) =  0.3173907536415453 5622.048 1579
slice = train, score = 0.3173907536415453
np.mean(results), mse, len(results) =  0.2678055555555555 5696.7637 720
slice = test, score = 0.2678055555555555

=== Iteration 21000 ===
np.mean(results), mse, len(results) =  0.25860000000000005 6343.205 100
slice = key, score = 0.25860000000000005
np.mean(results), mse, len(results) =  0.3211969601013299 5887.3154 1579
slice = train, score = 0.3211969601013299
np.mean(results), mse, len(results) =  0.2712638888888889 5943.473 720
slice = test, score = 0.2712638888888889

=== Iteration 22000 ===
np.mean(results), mse, len(results) =  0.2495 6929.6436 100
slice = key, score = 0.2495
np.mean(results), mse, len(results) =  0.31427485750474987 6439.553 1579
slice = train, score = 0.31427485750474987
np.mean(results), mse, len(results) =  0

np.mean(results), mse, len(results) =  0.3298226725775808 19770.484 1579
slice = train, score = 0.3298226725775808
np.mean(results), mse, len(results) =  0.2727083333333334 19899.08 720
slice = test, score = 0.2727083333333334

=== Iteration 44000 ===
np.mean(results), mse, len(results) =  0.25980000000000003 21338.258 100
slice = key, score = 0.25980000000000003
np.mean(results), mse, len(results) =  0.33234325522482583 20413.705 1579
slice = train, score = 0.33234325522482583
np.mean(results), mse, len(results) =  0.2793888888888889 20487.912 720
slice = test, score = 0.2793888888888889

=== Iteration 45000 ===
np.mean(results), mse, len(results) =  0.2653 22299.455 100
slice = key, score = 0.2653
np.mean(results), mse, len(results) =  0.3356491450284991 21381.932 1579
slice = train, score = 0.3356491450284991
np.mean(results), mse, len(results) =  0.28144444444444444 21547.84 720
slice = test, score = 0.28144444444444444

=== Iteration 46000 ===
np.mean(results), mse, len(results) =

np.mean(results), mse, len(results) =  0.2857638888888889 41579.207 720
slice = test, score = 0.2857638888888889

=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.2701 43720.63 100
slice = key, score = 0.2701
np.mean(results), mse, len(results) =  0.3416402786573781 42715.434 1579
slice = train, score = 0.3416402786573781
np.mean(results), mse, len(results) =  0.28577777777777774 42845.777 720
slice = test, score = 0.28577777777777774

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.2716 44677.7 100
slice = key, score = 0.2716
np.mean(results), mse, len(results) =  0.345573147561748 43705.734 1579
slice = train, score = 0.345573147561748
np.mean(results), mse, len(results) =  0.28841666666666665 43865.062 720
slice = test, score = 0.28841666666666665

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.2762 45967.89 100
slice = key, score = 0.2762
np.mean(results), mse, len(results) =  0.34406586447118426 44947.75 1579
slice = train, score 

np.mean(results), mse, len(results) =  0.35295123495883474 72701.72 1579
slice = train, score = 0.35295123495883474
np.mean(results), mse, len(results) =  0.2901111111111111 72876.336 720
slice = test, score = 0.2901111111111111

=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.2764 74835.48 100
slice = key, score = 0.2764
np.mean(results), mse, len(results) =  0.3528055731475618 74066.29 1579
slice = train, score = 0.3528055731475618
np.mean(results), mse, len(results) =  0.29079166666666667 74228.79 720
slice = test, score = 0.29079166666666667

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.2749 76466.8 100
slice = key, score = 0.2749
np.mean(results), mse, len(results) =  0.34783407219759344 75789.49 1579
slice = train, score = 0.34783407219759344
np.mean(results), mse, len(results) =  0.2855972222222222 75941.875 720
slice = test, score = 0.2855972222222222

=== Iteration 93000 ===
np.mean(results), mse, len(results) =  0.27940000000000004 77943.0

In [22]:
for dataset in DATASETS[-1:]:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 1000000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "learning_rate": 0.0005
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  military
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1700.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1100.pkl
Loading file military/ment_to_ent_scores_n

np.mean(results), mse, len(results) =  0.2464 4224.237 100
slice = key, score = 0.2464
np.mean(results), mse, len(results) =  0.3086763774540849 3827.488 1579
slice = train, score = 0.3086763774540849
np.mean(results), mse, len(results) =  0.2598611111111111 3875.1804 720
slice = test, score = 0.2598611111111111

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.24780000000000002 4649.822 100
slice = key, score = 0.24780000000000002
np.mean(results), mse, len(results) =  0.31376187460417987 4231.7744 1579
slice = train, score = 0.31376187460417987
np.mean(results), mse, len(results) =  0.2659722222222223 4273.015 720
slice = test, score = 0.2659722222222223

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.2511 5132.789 100
slice = key, score = 0.2511
np.mean(results), mse, len(results) =  0.31310956301456616 4690.0586 1579
slice = train, score = 0.31310956301456616
np.mean(results), mse, len(results) =  0.26545833333333335 4734.478 720
slice = test, scor

np.mean(results), mse, len(results) =  0.335718809373021 16635.43 1579
slice = train, score = 0.335718809373021
np.mean(results), mse, len(results) =  0.2837777777777778 16689.326 720
slice = test, score = 0.2837777777777778

=== Iteration 40000 ===
np.mean(results), mse, len(results) =  0.26269999999999993 18334.465 100
slice = key, score = 0.26269999999999993
np.mean(results), mse, len(results) =  0.3318429385687144 17573.531 1579
slice = train, score = 0.3318429385687144
np.mean(results), mse, len(results) =  0.2802777777777778 17624.287 720
slice = test, score = 0.2802777777777778

=== Iteration 41000 ===
np.mean(results), mse, len(results) =  0.2695 18996.176 100
slice = key, score = 0.2695
np.mean(results), mse, len(results) =  0.33369854338188726 18279.225 1579
slice = train, score = 0.33369854338188726
np.mean(results), mse, len(results) =  0.27868055555555554 18356.393 720
slice = test, score = 0.27868055555555554

=== Iteration 42000 ===
np.mean(results), mse, len(results) = 


=== Iteration 63000 ===
np.mean(results), mse, len(results) =  0.2728 39198.79 100
slice = key, score = 0.2728
np.mean(results), mse, len(results) =  0.3409689677010766 38433.656 1579
slice = train, score = 0.3409689677010766
np.mean(results), mse, len(results) =  0.2829305555555556 38525.867 720
slice = test, score = 0.2829305555555556

=== Iteration 64000 ===
np.mean(results), mse, len(results) =  0.27509999999999996 40324.383 100
slice = key, score = 0.27509999999999996
np.mean(results), mse, len(results) =  0.34149461684610516 39621.234 1579
slice = train, score = 0.34149461684610516
np.mean(results), mse, len(results) =  0.28544444444444445 39696.77 720
slice = test, score = 0.28544444444444445

=== Iteration 65000 ===
np.mean(results), mse, len(results) =  0.27430000000000004 41535.207 100
slice = key, score = 0.27430000000000004
np.mean(results), mse, len(results) =  0.34335655478150734 40665.47 1579
slice = train, score = 0.34335655478150734
np.mean(results), mse, len(results)

np.mean(results), mse, len(results) =  0.2950277777777778 67241.21 720
slice = test, score = 0.2950277777777778

=== Iteration 87000 ===
np.mean(results), mse, len(results) =  0.2843 69348.14 100
slice = key, score = 0.2843
np.mean(results), mse, len(results) =  0.3530652311589614 68621.17 1579
slice = train, score = 0.3530652311589614
np.mean(results), mse, len(results) =  0.2914444444444444 68699.08 720
slice = test, score = 0.2914444444444444

=== Iteration 88000 ===
np.mean(results), mse, len(results) =  0.28479999999999994 70476.55 100
slice = key, score = 0.28479999999999994
np.mean(results), mse, len(results) =  0.35672577580747306 69923.13 1579
slice = train, score = 0.35672577580747306
np.mean(results), mse, len(results) =  0.2938472222222222 69996.33 720
slice = test, score = 0.2938472222222222

=== Iteration 89000 ===
np.mean(results), mse, len(results) =  0.28370000000000006 72052.25 100
slice = key, score = 0.28370000000000006
np.mean(results), mse, len(results) =  0.35606

np.mean(results), mse, len(results) =  0.3616592780240659 105768.59 1579
slice = train, score = 0.3616592780240659
np.mean(results), mse, len(results) =  0.29743055555555553 105693.61 720
slice = test, score = 0.29743055555555553

=== Iteration 111000 ===
np.mean(results), mse, len(results) =  0.28490000000000004 107723.66 100
slice = key, score = 0.28490000000000004
np.mean(results), mse, len(results) =  0.3587650411652945 107228.48 1579
slice = train, score = 0.3587650411652945
np.mean(results), mse, len(results) =  0.29347222222222225 107238.555 720
slice = test, score = 0.29347222222222225

=== Iteration 112000 ===
np.mean(results), mse, len(results) =  0.28719999999999996 109484.17 100
slice = key, score = 0.28719999999999996
np.mean(results), mse, len(results) =  0.35993666877770736 109116.29 1579
slice = train, score = 0.35993666877770736
np.mean(results), mse, len(results) =  0.296 109044.516 720
slice = test, score = 0.296

=== Iteration 113000 ===
np.mean(results), mse, len(r


=== Iteration 134000 ===
np.mean(results), mse, len(results) =  0.28520000000000006 151544.12 100
slice = key, score = 0.28520000000000006
np.mean(results), mse, len(results) =  0.35697276757441415 151556.17 1579
slice = train, score = 0.35697276757441415
np.mean(results), mse, len(results) =  0.29183333333333333 151380.33 720
slice = test, score = 0.29183333333333333

=== Iteration 135000 ===
np.mean(results), mse, len(results) =  0.287 153540.92 100
slice = key, score = 0.287
np.mean(results), mse, len(results) =  0.3597276757441419 153813.67 1579
slice = train, score = 0.3597276757441419
np.mean(results), mse, len(results) =  0.2944583333333333 153583.39 720
slice = test, score = 0.2944583333333333

=== Iteration 136000 ===
np.mean(results), mse, len(results) =  0.28990000000000005 155623.78 100
slice = key, score = 0.28990000000000005
np.mean(results), mse, len(results) =  0.3636162127929069 155656.92 1579
slice = train, score = 0.3636162127929069
np.mean(results), mse, len(result

np.mean(results), mse, len(results) =  0.29581944444444447 200823.86 720
slice = test, score = 0.29581944444444447

=== Iteration 158000 ===
np.mean(results), mse, len(results) =  0.2911 202852.94 100
slice = key, score = 0.2911
np.mean(results), mse, len(results) =  0.3654211526282457 203433.77 1579
slice = train, score = 0.3654211526282457
np.mean(results), mse, len(results) =  0.29783333333333334 203084.73 720
slice = test, score = 0.29783333333333334

=== Iteration 159000 ===
np.mean(results), mse, len(results) =  0.2907 205024.06 100
slice = key, score = 0.2907
np.mean(results), mse, len(results) =  0.36580747308423045 205893.98 1579
slice = train, score = 0.36580747308423045
np.mean(results), mse, len(results) =  0.2991111111111111 205437.28 720
slice = test, score = 0.2991111111111111

=== Iteration 160000 ===
np.mean(results), mse, len(results) =  0.29660000000000003 207822.31 100
slice = key, score = 0.29660000000000003
np.mean(results), mse, len(results) =  0.3673210892970234

MemoryError: Unable to allocate 1.23 GiB for an array with shape (1579, 104520) and data type int64

In [15]:
for dataset in DATASETS[-1:]:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100
    )
    
    ctx.set_kmeans_games_as_key()
    
    N = 1000000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "learning_rate": 0.0005
    })
    
    m.fit()
    m.get_score("train"), m.get_score("test")
    
    
    del ctx
    gc.collect()

    del m
    gc.collect()




DATASET =  military
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1700.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1100.pkl
Loading file military/ment_to_ent_scores_n

np.mean(results), mse, len(results) =  0.263625 3955.2295 720
slice = test, score = 0.263625

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.24450000000000002 4671.435 100
slice = key, score = 0.24450000000000002
np.mean(results), mse, len(results) =  0.31177327422419254 4263.8496 1579
slice = train, score = 0.31177327422419254
np.mean(results), mse, len(results) =  0.2624444444444445 4324.819 720
slice = test, score = 0.2624444444444445

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.2501 4998.445 100
slice = key, score = 0.2501
np.mean(results), mse, len(results) =  0.31320455984800505 4661.4893 1579
slice = train, score = 0.31320455984800505
np.mean(results), mse, len(results) =  0.26325 4724.636 720
slice = test, score = 0.26325

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.2413 5594.0405 100
slice = key, score = 0.2413
np.mean(results), mse, len(results) =  0.3083090563647879 5143.836 1579
slice = train, score = 0.30830905636

np.mean(results), mse, len(results) =  0.3326852438252058 17770.13 1579
slice = train, score = 0.3326852438252058
np.mean(results), mse, len(results) =  0.27752777777777776 17916.293 720
slice = test, score = 0.27752777777777776

=== Iteration 41000 ===
np.mean(results), mse, len(results) =  0.2614 19214.117 100
slice = key, score = 0.2614
np.mean(results), mse, len(results) =  0.3328562381253958 18463.006 1579
slice = train, score = 0.3328562381253958
np.mean(results), mse, len(results) =  0.27799999999999997 18597.969 720
slice = test, score = 0.27799999999999997

=== Iteration 42000 ===
np.mean(results), mse, len(results) =  0.2603 19860.479 100
slice = key, score = 0.2603
np.mean(results), mse, len(results) =  0.33239392020265984 19145.426 1579
slice = train, score = 0.33239392020265984
np.mean(results), mse, len(results) =  0.27908333333333335 19271.45 720
slice = test, score = 0.27908333333333335

=== Iteration 43000 ===
np.mean(results), mse, len(results) =  0.2596 20930.355 100


=== Iteration 64000 ===
np.mean(results), mse, len(results) =  0.2723 40815.883 100
slice = key, score = 0.2723
np.mean(results), mse, len(results) =  0.34164661177960737 40005.117 1579
slice = train, score = 0.34164661177960737
np.mean(results), mse, len(results) =  0.28326388888888887 40169.88 720
slice = test, score = 0.28326388888888887

=== Iteration 65000 ===
np.mean(results), mse, len(results) =  0.2677 42041.863 100
slice = key, score = 0.2677
np.mean(results), mse, len(results) =  0.34135528815706145 41100.77 1579
slice = train, score = 0.34135528815706145
np.mean(results), mse, len(results) =  0.281625 41176.52 720
slice = test, score = 0.281625

=== Iteration 66000 ===
np.mean(results), mse, len(results) =  0.2773 42787.934 100
slice = key, score = 0.2773
np.mean(results), mse, len(results) =  0.34842305256491446 41926.816 1579
slice = train, score = 0.34842305256491446
np.mean(results), mse, len(results) =  0.2878472222222222 42008.62 720
slice = test, score = 0.2878472222

np.mean(results), mse, len(results) =  0.2842361111111111 68812.734 720
slice = test, score = 0.2842361111111111

=== Iteration 88000 ===
np.mean(results), mse, len(results) =  0.2817 70844.34 100
slice = key, score = 0.2817
np.mean(results), mse, len(results) =  0.3535782140595314 70163.92 1579
slice = train, score = 0.3535782140595314
np.mean(results), mse, len(results) =  0.29141666666666666 70192.35 720
slice = test, score = 0.29141666666666666

=== Iteration 89000 ===
np.mean(results), mse, len(results) =  0.28130000000000005 72301.68 100
slice = key, score = 0.28130000000000005
np.mean(results), mse, len(results) =  0.3548828372387588 71554.74 1579
slice = train, score = 0.3548828372387588
np.mean(results), mse, len(results) =  0.2904027777777778 71703.67 720
slice = test, score = 0.2904027777777778

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.2802 73523.87 100
slice = key, score = 0.2802
np.mean(results), mse, len(results) =  0.35552881570614303 72852.81 157

...

np.mean(results), mse, len(results) =  0.2994027777777778 2192985.8 720
slice = test, score = 0.2994027777777778

=== Iteration 886000 ===
np.mean(results), mse, len(results) =  0.3052 2135094.5 100
slice = key, score = 0.3052
np.mean(results), mse, len(results) =  0.37951234958834706 2223337.5 1579
slice = train, score = 0.37951234958834706
np.mean(results), mse, len(results) =  0.29904166666666665 2197658.2 720
slice = test, score = 0.29904166666666665

=== Iteration 887000 ===
np.mean(results), mse, len(results) =  0.29950000000000004 2137074.2 100
slice = key, score = 0.29950000000000004
np.mean(results), mse, len(results) =  0.37994933502216593 2226340.2 1579
slice = train, score = 0.37994933502216593
np.mean(results), mse, len(results) =  0.29812500000000003 2200838.8 720
slice = test, score = 0.29812500000000003

=== Iteration 888000 ===
np.mean(results), mse, len(results) =  0.3062 2129199.2 100
slice = key, score = 0.3062
np.mean(results), mse, len(results) =  0.38245091830272


=== Iteration 909000 ===
np.mean(results), mse, len(results) =  0.30699999999999994 2200083.5 100
slice = key, score = 0.30699999999999994
np.mean(results), mse, len(results) =  0.3826282457251425 2284030.0 1579
slice = train, score = 0.3826282457251425
np.mean(results), mse, len(results) =  0.30109722222222224 2267879.0 720
slice = test, score = 0.30109722222222224

=== Iteration 910000 ===
np.mean(results), mse, len(results) =  0.3021 2197969.8 100
slice = key, score = 0.3021
np.mean(results), mse, len(results) =  0.3797403419886004 2286090.5 1579
slice = train, score = 0.3797403419886004
np.mean(results), mse, len(results) =  0.2975972222222222 2264078.0 720
slice = test, score = 0.2975972222222222

=== Iteration 911000 ===
np.mean(results), mse, len(results) =  0.3074 2201522.5 100
slice = key, score = 0.3074
np.mean(results), mse, len(results) =  0.38102596580113995 2292141.0 1579
slice = train, score = 0.38102596580113995
np.mean(results), mse, len(results) =  0.2986805555555555

np.mean(results), mse, len(results) =  0.2992361111111111 2323977.5 720
slice = test, score = 0.2992361111111111

=== Iteration 933000 ===
np.mean(results), mse, len(results) =  0.30569999999999997 2250149.5 100
slice = key, score = 0.30569999999999997
np.mean(results), mse, len(results) =  0.3844711842938569 2351767.5 1579
slice = train, score = 0.3844711842938569
np.mean(results), mse, len(results) =  0.29766666666666663 2322865.0 720
slice = test, score = 0.29766666666666663

=== Iteration 934000 ===
np.mean(results), mse, len(results) =  0.3038 2257141.0 100
slice = key, score = 0.3038
np.mean(results), mse, len(results) =  0.3807979734008866 2355000.8 1579
slice = train, score = 0.3807979734008866
np.mean(results), mse, len(results) =  0.2956388888888889 2327468.0 720
slice = test, score = 0.2956388888888889

=== Iteration 935000 ===
np.mean(results), mse, len(results) =  0.302 2256054.0 100
slice = key, score = 0.302
np.mean(results), mse, len(results) =  0.3799430018999366 23545


=== Iteration 956000 ===
np.mean(results), mse, len(results) =  0.3022 2318468.2 100
slice = key, score = 0.3022
np.mean(results), mse, len(results) =  0.38151361621279295 2415836.8 1579
slice = train, score = 0.38151361621279295
np.mean(results), mse, len(results) =  0.2959444444444444 2386513.8 720
slice = test, score = 0.2959444444444444

=== Iteration 957000 ===
np.mean(results), mse, len(results) =  0.3066 2317307.2 100
slice = key, score = 0.3066
np.mean(results), mse, len(results) =  0.3855541481950601 2418608.0 1579
slice = train, score = 0.3855541481950601
np.mean(results), mse, len(results) =  0.300875 2391426.2 720
slice = test, score = 0.300875

=== Iteration 958000 ===
np.mean(results), mse, len(results) =  0.30690000000000006 2318149.8 100
slice = key, score = 0.30690000000000006
np.mean(results), mse, len(results) =  0.3844838505383154 2419371.8 1579
slice = train, score = 0.3844838505383154
np.mean(results), mse, len(results) =  0.29854166666666665 2390950.8 720
slice 

np.mean(results), mse, len(results) =  0.3006111111111111 2451211.0 720
slice = test, score = 0.3006111111111111

=== Iteration 980000 ===
np.mean(results), mse, len(results) =  0.3098 2380729.2 100
slice = key, score = 0.3098
np.mean(results), mse, len(results) =  0.3845281823939202 2482633.0 1579
slice = train, score = 0.3845281823939202
np.mean(results), mse, len(results) =  0.3001111111111111 2453148.8 720
slice = test, score = 0.3001111111111111

=== Iteration 981000 ===
np.mean(results), mse, len(results) =  0.30260000000000004 2374200.8 100
slice = key, score = 0.30260000000000004
np.mean(results), mse, len(results) =  0.3828815706143129 2485625.0 1579
slice = train, score = 0.3828815706143129
np.mean(results), mse, len(results) =  0.2972777777777778 2452775.8 720
slice = test, score = 0.2972777777777778

=== Iteration 982000 ===
np.mean(results), mse, len(results) =  0.30069999999999997 2379154.0 100
slice = key, score = 0.30069999999999997
np.mean(results), mse, len(results) =

In [17]:
import torch

In [62]:
ctx = EvalContextRelevs(
    load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
    det_attempts=100
)

ctx.set_kmeans_games_as_key()


ev([
Popular(ctx),
AnnCUR(ctx),
])

Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1600.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0800.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1700.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1100.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_l

KeyboardInterrupt: 

In [75]:
s = """
=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0443 35.883972 100
slice = key, score = 0.0443
np.mean(results), mse, len(results) =  0.047946902654867264 32.904503 2260
slice = train, score = 0.047946902654867264
np.mean(results), mse, len(results) =  0.04920039486673248 33.015144 1013
slice = test, score = 0.04920039486673248

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5117999999999999 86.57781 100
slice = key, score = 0.5117999999999999
np.mean(results), mse, len(results) =  0.5132654867256637 86.5766 2260
slice = train, score = 0.5132654867256637
np.mean(results), mse, len(results) =  0.48496544916090817 85.01465 1013
slice = test, score = 0.48496544916090817

=== Iteration 2000 ===
np.mean(results), mse, len(results) =  0.5356000000000001 136.45412 100
slice = key, score = 0.5356000000000001
np.mean(results), mse, len(results) =  0.5406991150442478 135.13806 2260
slice = train, score = 0.5406991150442478
np.mean(results), mse, len(results) =  0.5041263573543929 134.2659 1013
slice = test, score = 0.5041263573543929

=== Iteration 4000 ===
np.mean(results), mse, len(results) =  0.5546 261.0547 100
slice = key, score = 0.5546
np.mean(results), mse, len(results) =  0.5607743362831858 259.49985 2260
slice = train, score = 0.5607743362831858
np.mean(results), mse, len(results) =  0.5173050345508391 258.06662 1013
slice = test, score = 0.5173050345508391

=== Iteration 5000 ===
np.mean(results), mse, len(results) =  0.5565 336.81384 100
slice = key, score = 0.5565
np.mean(results), mse, len(results) =  0.5636283185840709 334.65643 2260
slice = train, score = 0.5636283185840709
np.mean(results), mse, len(results) =  0.5170187561697928 333.33746 1013
slice = test, score = 0.5170187561697928

=== Iteration 6000 ===
np.mean(results), mse, len(results) =  0.5591 412.82468 100
slice = key, score = 0.5591
np.mean(results), mse, len(results) =  0.5678628318584071 407.82117 2260
slice = train, score = 0.5678628318584071
np.mean(results), mse, len(results) =  0.5192398815399802 405.96143 1013
slice = test, score = 0.5192398815399802

=== Iteration 7000 ===
np.mean(results), mse, len(results) =  0.5645 497.19446 100
slice = key, score = 0.5645
np.mean(results), mse, len(results) =  0.5747566371681416 493.84232 2260
slice = train, score = 0.5747566371681416
np.mean(results), mse, len(results) =  0.5235241855873644 491.8354 1013
slice = test, score = 0.5235241855873644

=== Iteration 8000 ===
np.mean(results), mse, len(results) =  0.5599999999999999 589.84357 100
slice = key, score = 0.5599999999999999
np.mean(results), mse, len(results) =  0.5734911504424779 585.05896 2260
slice = train, score = 0.5734911504424779
np.mean(results), mse, len(results) =  0.520720631786772 583.53217 1013
slice = test, score = 0.520720631786772

=== Iteration 9000 ===
np.mean(results), mse, len(results) =  0.5708 662.2139 100
slice = key, score = 0.5708
np.mean(results), mse, len(results) =  0.5795221238938053 654.68286 2260
slice = train, score = 0.5795221238938053
np.mean(results), mse, len(results) =  0.523919052319842 652.5642 1013
slice = test, score = 0.523919052319842

=== Iteration 10000 ===
np.mean(results), mse, len(results) =  0.5685 767.95886 100
slice = key, score = 0.5685
np.mean(results), mse, len(results) =  0.577141592920354 761.63055 2260
slice = train, score = 0.577141592920354
np.mean(results), mse, len(results) =  0.5238894373149062 759.5815 1013
slice = test, score = 0.5238894373149062

=== Iteration 11000 ===
np.mean(results), mse, len(results) =  0.5685 863.0076 100
slice = key, score = 0.5685
np.mean(results), mse, len(results) =  0.5776548672566372 852.02484 2260
slice = train, score = 0.5776548672566372
np.mean(results), mse, len(results) =  0.5240375123395854 849.4952 1013
slice = test, score = 0.5240375123395854

=== Iteration 12000 ===
np.mean(results), mse, len(results) =  0.5719000000000001 964.3046 100
slice = key, score = 0.5719000000000001
np.mean(results), mse, len(results) =  0.5801991150442478 956.99347 2260
slice = train, score = 0.5801991150442478
np.mean(results), mse, len(results) =  0.5249654491609083 953.16943 1013
slice = test, score = 0.5249654491609083

=== Iteration 13000 ===
np.mean(results), mse, len(results) =  0.5742 1051.4691 100
slice = key, score = 0.5742
np.mean(results), mse, len(results) =  0.5871017699115044 1041.1113 2260
slice = train, score = 0.5871017699115044
np.mean(results), mse, len(results) =  0.5281737413622902 1039.6813 1013
slice = test, score = 0.5281737413622902

=== Iteration 14000 ===
np.mean(results), mse, len(results) =  0.5741999999999998 1156.0157 100
slice = key, score = 0.5741999999999998
np.mean(results), mse, len(results) =  0.5876991150442478 1150.5294 2260
slice = train, score = 0.5876991150442478
np.mean(results), mse, len(results) =  0.5281342546890424 1149.726 1013
slice = test, score = 0.5281342546890424

=== Iteration 15000 ===
np.mean(results), mse, len(results) =  0.5806 1271.3711 100
slice = key, score = 0.5806
np.mean(results), mse, len(results) =  0.5921858407079647 1257.8185 2260
slice = train, score = 0.5921858407079647
np.mean(results), mse, len(results) =  0.5313228035538006 1258.2802 1013
slice = test, score = 0.5313228035538006

=== Iteration 16000 ===
np.mean(results), mse, len(results) =  0.5802999999999999 1339.0886 100
slice = key, score = 0.5802999999999999
np.mean(results), mse, len(results) =  0.5919778761061947 1329.4121 2260
slice = train, score = 0.5919778761061947
np.mean(results), mse, len(results) =  0.5303257650542942 1330.3677 1013
slice = test, score = 0.5303257650542942

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.5801 1441.9502 100
slice = key, score = 0.5801
np.mean(results), mse, len(results) =  0.594504424778761 1432.3976 2260
slice = train, score = 0.594504424778761
np.mean(results), mse, len(results) =  0.5301085883514314 1432.3578 1013
slice = test, score = 0.5301085883514314

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5812 1534.8604 100
slice = key, score = 0.5812
np.mean(results), mse, len(results) =  0.5951194690265487 1529.9674 2260
slice = train, score = 0.5951194690265487
np.mean(results), mse, len(results) =  0.53 1528.3907 1013
slice = test, score = 0.53

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.5833 1623.5963 100
slice = key, score = 0.5833
np.mean(results), mse, len(results) =  0.5967212389380532 1610.1062 2260
slice = train, score = 0.5967212389380532
np.mean(results), mse, len(results) =  0.5315103652517276 1608.2421 1013
slice = test, score = 0.5315103652517276

=== Iteration 20000 ===
np.mean(results), mse, len(results) =  0.5824000000000001 1710.7817 100
slice = key, score = 0.5824000000000001
np.mean(results), mse, len(results) =  0.5993407079646018 1703.5121 2260
slice = train, score = 0.5993407079646018
np.mean(results), mse, len(results) =  0.5326061204343534 1698.2251 1013
slice = test, score = 0.5326061204343534

=== Iteration 21000 ===
np.mean(results), mse, len(results) =  0.5840000000000001 1812.0786 100
slice = key, score = 0.5840000000000001

np.mean(results), mse, len(results) =  0.5955132743362832 1802.1031 2260
slice = train, score = 0.5955132743362832
np.mean(results), mse, len(results) =  0.531253701875617 1799.7714 1013
slice = test, score = 0.531253701875617

=== Iteration 22000 ===
np.mean(results), mse, len(results) =  0.5858 1928.6364 100
slice = key, score = 0.5858
np.mean(results), mse, len(results) =  0.6023849557522123 1918.6433 2260
slice = train, score = 0.6023849557522123
np.mean(results), mse, len(results) =  0.5325074037512341 1921.6398 1013
slice = test, score = 0.5325074037512341

=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.5863 2021.5597 100
slice = key, score = 0.5863
np.mean(results), mse, len(results) =  0.6014778761061946 2012.535 2260
slice = train, score = 0.6014778761061946
np.mean(results), mse, len(results) =  0.534284304047384 2014.825 1013
slice = test, score = 0.534284304047384

=== Iteration 24000 ===
np.mean(results), mse, len(results) =  0.5871 2119.527 100
slice = key, score = 0.5871
np.mean(results), mse, len(results) =  0.6001283185840708 2109.5823 2260
slice = train, score = 0.6001283185840708
np.mean(results), mse, len(results) =  0.5322211253701875 2114.1206 1013
slice = test, score = 0.5322211253701875

=== Iteration 25000 ===
np.mean(results), mse, len(results) =  0.5890000000000001 2227.9634 100
slice = key, score = 0.5890000000000001
np.mean(results), mse, len(results) =  0.6052345132743363 2208.0276 2260
slice = train, score = 0.6052345132743363
np.mean(results), mse, len(results) =  0.533731490621915 2212.5608 1013
slice = test, score = 0.533731490621915

=== Iteration 26000 ===
np.mean(results), mse, len(results) =  0.5888000000000001 2330.915 100
slice = key, score = 0.5888000000000001
np.mean(results), mse, len(results) =  0.6044380530973451 2315.9692 2260
slice = train, score = 0.6044380530973451
np.mean(results), mse, len(results) =  0.5344323790720632 2317.2888 1013
slice = test, score = 0.5344323790720632

=== Iteration 27000 ===
np.mean(results), mse, len(results) =  0.5892000000000001 2422.9614 100
slice = key, score = 0.5892000000000001
np.mean(results), mse, len(results) =  0.6069292035398229 2412.089 2260
slice = train, score = 0.6069292035398229
np.mean(results), mse, len(results) =  0.5355972359328727 2415.4448 1013
slice = test, score = 0.5355972359328727

=== Iteration 28000 ===
np.mean(results), mse, len(results) =  0.5895 2507.7854 100
slice = key, score = 0.5895
np.mean(results), mse, len(results) =  0.607424778761062 2510.2234 2260
slice = train, score = 0.607424778761062
np.mean(results), mse, len(results) =  0.5341855873642646 2512.2358 1013
slice = test, score = 0.5341855873642646

=== Iteration 29000 ===
np.mean(results), mse, len(results) =  0.59 2590.3977 100
slice = key, score = 0.59
np.mean(results), mse, len(results) =  0.6063495575221238 2582.9756 2260
slice = train, score = 0.6063495575221238
np.mean(results), mse, len(results) =  0.5329812438302073 2585.9878 1013
slice = test, score = 0.5329812438302073

=== Iteration 30000 ===
np.mean(results), mse, len(results) =  0.5925 2740.2935 100
slice = key, score = 0.5925
np.mean(results), mse, len(results) =  0.6100442477876106 2726.8748 2260
slice = train, score = 0.6100442477876106
np.mean(results), mse, len(results) =  0.5361895360315894 2733.4102 1013
slice = test, score = 0.5361895360315894

=== Iteration 31000 ===
np.mean(results), mse, len(results) =  0.5958000000000001 2789.9001 100
slice = key, score = 0.5958000000000001
np.mean(results), mse, len(results) =  0.61 2776.8574 2260
slice = train, score = 0.61
np.mean(results), mse, len(results) =  0.5349753208292201 2798.1694 1013
slice = test, score = 0.5349753208292201

=== Iteration 32000 ===
np.mean(results), mse, len(results) =  0.5985 2883.3755 100
slice = key, score = 0.5985
np.mean(results), mse, len(results) =  0.609141592920354 2880.1824 2260
slice = train, score = 0.609141592920354
np.mean(results), mse, len(results) =  0.5345212240868707 2894.2595 1013
slice = test, score = 0.5345212240868707

=== Iteration 33000 ===
np.mean(results), mse, len(results) =  0.5982 2965.1003 100
slice = key, score = 0.5982
np.mean(results), mse, len(results) =  0.6122123893805309 2953.0818 2260
slice = train, score = 0.6122123893805309
np.mean(results), mse, len(results) =  0.5369002961500493 2964.0735 1013
slice = test, score = 0.5369002961500493

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5986 3062.9575 100
slice = key, score = 0.5986
np.mean(results), mse, len(results) =  0.6111327433628319 3058.6335 2260
slice = train, score = 0.6111327433628319
np.mean(results), mse, len(results) =  0.5352221125370188 3077.376 1013
slice = test, score = 0.5352221125370188

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.6003000000000001 3167.982 100
slice = key, score = 0.6003000000000001
np.mean(results), mse, len(results) =  0.6149690265486726 3157.5737 2260
slice = train, score = 0.6149690265486726
np.mean(results), mse, len(results) =  0.5361895360315894 3178.1755 1013
slice = test, score = 0.5361895360315894

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.5966000000000001 3289.7449 100
slice = key, score = 0.5966000000000001
np.mean(results), mse, len(results) =  0.6126725663716813 3271.506 2260
slice = train, score = 0.6126725663716813
np.mean(results), mse, len(results) =  0.5366041461006911 3294.2034 1013
slice = test, score = 0.5366041461006911

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.6013000000000001 3336.869 100
slice = key, score = 0.6013000000000001
np.mean(results), mse, len(results) =  0.6149203539823009 3326.6843 2260
slice = train, score = 0.6149203539823009
np.mean(results), mse, len(results) =  0.5371865745310958 3343.5388 1013
slice = test, score = 0.5371865745310958

=== Iteration 38000 ===
np.mean(results), mse, len(results) =  0.5931000000000001 3454.758 100
slice = key, score = 0.5931000000000001
np.mean(results), mse, len(results) =  0.6126814159292034 3432.2817 2260
slice = train, score = 0.6126814159292034
np.mean(results), mse, len(results) =  0.5347285291214215 3447.0583 1013
slice = test, score = 0.5347285291214215

=== Iteration 39000 ===
np.mean(results), mse, len(results) =  0.6012000000000001 3543.906 100
slice = key, score = 0.6012000000000001
np.mean(results), mse, len(results) =  0.6147566371681416 3529.9893 2260
slice = train, score = 0.6147566371681416
np.mean(results), mse, len(results) =  0.5349555774925964 3549.7983 1013
slice = test, score = 0.5349555774925964

=== Iteration 40000 ===
np.mean(results), mse, len(results) =  0.6024 3602.234 100
slice = key, score = 0.6024
np.mean(results), mse, len(results) =  0.6165530973451326 3600.616 2260
slice = train, score = 0.6165530973451326
np.mean(results), mse, len(results) =  0.535725567620928 3631.3125 1013
slice = test, score = 0.535725567620928

=== Iteration 41000 ===
np.mean(results), mse, len(results) =  0.5995 3688.5193 100
slice = key, score = 0.5995
np.mean(results), mse, len(results) =  0.6158318584070797 3674.0322 2260
slice = train, score = 0.6158318584070797
np.mean(results), mse, len(results) =  0.5376307996051334 3693.5945 1013
slice = test, score = 0.5376307996051334

=== Iteration 42000 ===
np.mean(results), mse, len(results) =  0.6048 3796.7375 100
slice = key, score = 0.6048
np.mean(results), mse, len(results) =  0.6198849557522124 3770.7756 2260
slice = train, score = 0.6198849557522124
np.mean(results), mse, len(results) =  0.5396742349457058 3782.829 1013
slice = test, score = 0.5396742349457058

=== Iteration 43000 ===
np.mean(results), mse, len(results) =  0.6026 3902.2866 100
slice = key, score = 0.6026
np.mean(results), mse, len(results) =  0.6189601769911504 3883.375 2260
slice = train, score = 0.6189601769911504
np.mean(results), mse, len(results) =  0.5382724580454097 3899.4346 1013
slice = test, score = 0.5382724580454097

=== Iteration 44000 ===
np.mean(results), mse, len(results) =  0.6053000000000001 3958.637 100
slice = key, score = 0.6053000000000001
np.mean(results), mse, len(results) =  0.6204247787610618 3945.8552 2260
slice = train, score = 0.6204247787610618
np.mean(results), mse, len(results) =  0.5398617966436327 3970.0242 1013
slice = test, score = 0.5398617966436327


=== Iteration 45000 ===
np.mean(results), mse, len(results) =  0.5994000000000002 4066.5786 100
slice = key, score = 0.5994000000000002
np.mean(results), mse, len(results) =  0.6174424778761062 4038.3374 2260
slice = train, score = 0.6174424778761062
np.mean(results), mse, len(results) =  0.5362783810463968 4071.2786 1013
slice = test, score = 0.5362783810463968

=== Iteration 46000 ===
np.mean(results), mse, len(results) =  0.6038000000000001 4181.268 100
slice = key, score = 0.6038000000000001
np.mean(results), mse, len(results) =  0.6212743362831858 4146.2344 2260
slice = train, score = 0.6212743362831858
np.mean(results), mse, len(results) =  0.5373445212240868 4171.926 1013
slice = test, score = 0.5373445212240868

=== Iteration 47000 ===
np.mean(results), mse, len(results) =  0.6042 4273.632 100
slice = key, score = 0.6042
np.mean(results), mse, len(results) =  0.619079646017699 4233.189 2260
slice = train, score = 0.619079646017699
np.mean(results), mse, len(results) =  0.5383613030602172 4264.881 1013
slice = test, score = 0.5383613030602172

=== Iteration 48000 ===
np.mean(results), mse, len(results) =  0.6043999999999999 4316.7 100
slice = key, score = 0.6043999999999999
np.mean(results), mse, len(results) =  0.6207654867256638 4300.4697 2260
slice = train, score = 0.6207654867256638
np.mean(results), mse, len(results) =  0.53857847976308 4325.6714 1013
slice = test, score = 0.53857847976308

=== Iteration 49000 ===
np.mean(results), mse, len(results) =  0.6095999999999999 4421.915 100
slice = key, score = 0.6095999999999999
np.mean(results), mse, len(results) =  0.6226327433628319 4401.1123 2260
slice = train, score = 0.6226327433628319
np.mean(results), mse, len(results) =  0.5396939782823298 4438.191 1013
slice = test, score = 0.5396939782823298

=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.6085 4520.9546 100
slice = key, score = 0.6085
np.mean(results), mse, len(results) =  0.6246061946902656 4494.545 2260
slice = train, score = 0.6246061946902656
np.mean(results), mse, len(results) =  0.5398617966436328 4525.459 1013
slice = test, score = 0.5398617966436328

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.6061 4649.5723 100
slice = key, score = 0.6061
np.mean(results), mse, len(results) =  0.6196681415929204 4620.6685 2260
slice = train, score = 0.6196681415929204
np.mean(results), mse, len(results) =  0.5365153010858835 4657.0684 1013
slice = test, score = 0.5365153010858835

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.6064 4760.4316 100
slice = key, score = 0.6064
np.mean(results), mse, len(results) =  0.6205088495575221 4714.236 2260
slice = train, score = 0.6205088495575221
np.mean(results), mse, len(results) =  0.5363968410661402 4756.9316 1013
slice = test, score = 0.5363968410661402

=== Iteration 53000 ===
np.mean(results), mse, len(results) =  0.6061 4840.99 100
slice = key, score = 0.6061
np.mean(results), mse, len(results) =  0.619924778761062 4811.62 2260
slice = train, score = 0.619924778761062
np.mean(results), mse, len(results) =  0.535705824284304 4842.4497 1013
slice = test, score = 0.535705824284304

=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.6109 4919.8916 100
slice = key, score = 0.6109
np.mean(results), mse, len(results) =  0.6212345132743363 4904.702 2260
slice = train, score = 0.6212345132743363
np.mean(results), mse, len(results) =  0.536623889437315 4930.43 1013
slice = test, score = 0.536623889437315

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.6113000000000001 5028.456 100
slice = key, score = 0.6113000000000001
np.mean(results), mse, len(results) =  0.625575221238938 5004.753 2260
slice = train, score = 0.625575221238938
np.mean(results), mse, len(results) =  0.5404244817374136 5034.384 1013
slice = test, score = 0.5404244817374136

=== Iteration 56000 ===
np.mean(results), mse, len(results) =  0.6085 5140.299 100
slice = key, score = 0.6085
np.mean(results), mse, len(results) =  0.6212831858407081 5117.9316 2260
slice = train, score = 0.6212831858407081
np.mean(results), mse, len(results) =  0.5377295162882527 5158.8755 1013
slice = test, score = 0.5377295162882527

=== Iteration 57000 ===
np.mean(results), mse, len(results) =  0.6119 5264.2583 100
slice = key, score = 0.6119
np.mean(results), mse, len(results) =  0.622637168141593 5236.0527 2260
slice = train, score = 0.622637168141593
np.mean(results), mse, len(results) =  0.5369299111549852 5271.819 1013
slice = test, score = 0.5369299111549852

=== Iteration 58000 ===
np.mean(results), mse, len(results) =  0.6126 5312.72 100
slice = key, score = 0.6126
np.mean(results), mse, len(results) =  0.6254424778761062 5274.8364 2260
slice = train, score = 0.6254424778761062
np.mean(results), mse, len(results) =  0.5387956564659427 5317.0503 1013
slice = test, score = 0.5387956564659427

=== Iteration 59000 ===
np.mean(results), mse, len(results) =  0.614 5371.714 100
slice = key, score = 0.614
np.mean(results), mse, len(results) =  0.6249778761061947 5338.0293 2260
slice = train, score = 0.6249778761061947
np.mean(results), mse, len(results) =  0.5401085883514314 5373.5186 1013
slice = test, score = 0.5401085883514314

=== Iteration 60000 ===
np.mean(results), mse, len(results) =  0.6193 5419.158 100
slice = key, score = 0.6193
np.mean(results), mse, len(results) =  0.629 5415.071 2260
slice = train, score = 0.629
np.mean(results), mse, len(results) =  0.5384106614017768 5455.2666 1013
slice = test, score = 0.5384106614017768

=== Iteration 61000 ===
np.mean(results), mse, len(results) =  0.6176 5500.3086 100
slice = key, score = 0.6176
np.mean(results), mse, len(results) =  0.6279557522123894 5470.632 2260
slice = train, score = 0.6279557522123894
np.mean(results), mse, len(results) =  0.5391411648568609 5516.888 1013
slice = test, score = 0.5391411648568609

=== Iteration 62000 ===
np.mean(results), mse, len(results) =  0.6104999999999999 5579.8604 100
slice = key, score = 0.6104999999999999
np.mean(results), mse, len(results) =  0.6242699115044248 5570.439 2260
slice = train, score = 0.6242699115044248
np.mean(results), mse, len(results) =  0.5359032576505429 5622.6504 1013
slice = test, score = 0.5359032576505429

=== Iteration 63000 ===
np.mean(results), mse, len(results) =  0.6167 5718.763 100
slice = key, score = 0.6167
np.mean(results), mse, len(results) =  0.6291814159292035 5685.498 2260
slice = train, score = 0.6291814159292035
np.mean(results), mse, len(results) =  0.5383514313919053 5736.523 1013
slice = test, score = 0.5383514313919053

=== Iteration 64000 ===
np.mean(results), mse, len(results) =  0.6166 5752.1323 100
slice = key, score = 0.6166
np.mean(results), mse, len(results) =  0.6276061946902656 5720.106 2260
slice = train, score = 0.6276061946902656
np.mean(results), mse, len(results) =  0.5390128331688055 5762.1577 1013
slice = test, score = 0.5390128331688055

=== Iteration 65000 ===
np.mean(results), mse, len(results) =  0.6144000000000001 5842.087 100
slice = key, score = 0.6144000000000001
np.mean(results), mse, len(results) =  0.6281460176991152 5809.732 2260
slice = train, score = 0.6281460176991152
np.mean(results), mse, len(results) =  0.5391609081934847 5858.6016 1013
slice = test, score = 0.5391609081934847

=== Iteration 66000 ===
np.mean(results), mse, len(results) =  0.6170000000000001 5896.5522 100
slice = key, score = 0.6170000000000001
np.mean(results), mse, len(results) =  0.6277743362831858 5885.8735 2260
slice = train, score = 0.6277743362831858
np.mean(results), mse, len(results) =  0.5382132280355381 5934.3896 1013
slice = test, score = 0.5382132280355381

=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.6166 6021.805 100
slice = key, score = 0.6166
np.mean(results), mse, len(results) =  0.6293451327433628 6006.995 2260
slice = train, score = 0.6293451327433628
np.mean(results), mse, len(results) =  0.5385488647581441 6059.928 1013
slice = test, score = 0.5385488647581441

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.6138999999999999 6054.376 100
slice = key, score = 0.6138999999999999
np.mean(results), mse, len(results) =  0.627641592920354 6041.323 2260
slice = train, score = 0.627641592920354

np.mean(results), mse, len(results) =  0.53928923988154 6082.587 1013
slice = test, score = 0.53928923988154

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.6218 6161.571 100
slice = key, score = 0.6218
np.mean(results), mse, len(results) =  0.6322610619469026 6142.9185 2260
slice = train, score = 0.6322610619469026
np.mean(results), mse, len(results) =  0.5405923000987167 6205.284 1013
slice = test, score = 0.5405923000987167

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.6195 6237.225 100
slice = key, score = 0.6195
np.mean(results), mse, len(results) =  0.628637168141593 6203.5786 2260
slice = train, score = 0.628637168141593
np.mean(results), mse, len(results) =  0.537877591312932 6247.983 1013
slice = test, score = 0.537877591312932

=== Iteration 71000 ===
np.mean(results), mse, len(results) =  0.6181 6312.318 100
slice = key, score = 0.6181
np.mean(results), mse, len(results) =  0.6306061946902655 6300.797 2260
slice = train, score = 0.6306061946902655
np.mean(results), mse, len(results) =  0.5397038499506417 6346.8247 1013
slice = test, score = 0.5397038499506417

=== Iteration 72000 ===
np.mean(results), mse, len(results) =  0.6189 6407.0186 100
slice = key, score = 0.6189
np.mean(results), mse, len(results) =  0.6314070796460177 6373.7246 2260
slice = train, score = 0.6314070796460177
np.mean(results), mse, len(results) =  0.5391905231984205 6424.7563 1013
slice = test, score = 0.5391905231984205

=== Iteration 73000 ===
np.mean(results), mse, len(results) =  0.6214000000000001 6498.0454 100
slice = key, score = 0.6214000000000001
np.mean(results), mse, len(results) =  0.631858407079646 6507.1094 2260
slice = train, score = 0.631858407079646
np.mean(results), mse, len(results) =  0.5393780848963474 6557.9805 1013
slice = test, score = 0.5393780848963474

=== Iteration 74000 ===
np.mean(results), mse, len(results) =  0.6189999999999999 6516.8 100
slice = key, score = 0.6189999999999999
np.mean(results), mse, len(results) =  0.6331238938053098 6507.4917 2260
slice = train, score = 0.6331238938053098
np.mean(results), mse, len(results) =  0.5398617966436329 6555.178 1013
slice = test, score = 0.5398617966436329

=== Iteration 75000 ===
np.mean(results), mse, len(results) =  0.6216 6712.7656 100
slice = key, score = 0.6216
np.mean(results), mse, len(results) =  0.6318053097345133 6697.9434 2260
slice = train, score = 0.6318053097345133
np.mean(results), mse, len(results) =  0.5390128331688055 6758.386 1013
slice = test, score = 0.5390128331688055

=== Iteration 76000 ===
np.mean(results), mse, len(results) =  0.618 6751.7256 100
slice = key, score = 0.618
np.mean(results), mse, len(results) =  0.6300486725663716 6734.298 2260
slice = train, score = 0.6300486725663716
np.mean(results), mse, len(results) =  0.5393978282329713 6790.46 1013
slice = test, score = 0.5393978282329713

=== Iteration 77000 ===
np.mean(results), mse, len(results) =  0.6242000000000001 6800.2476 100
slice = key, score = 0.6242000000000001
np.mean(results), mse, len(results) =  0.6315044247787611 6796.8066 2260
slice = train, score = 0.6315044247787611
np.mean(results), mse, len(results) =  0.5389536031589339 6851.7056 1013
slice = test, score = 0.5389536031589339

=== Iteration 78000 ===
np.mean(results), mse, len(results) =  0.6215 6894.056 100
slice = key, score = 0.6215
np.mean(results), mse, len(results) =  0.6320486725663718 6892.982 2260
slice = train, score = 0.6320486725663718
np.mean(results), mse, len(results) =  0.54 6937.447 1013
slice = test, score = 0.54

=== Iteration 79000 ===
np.mean(results), mse, len(results) =  0.6224999999999999 6959.7456 100
slice = key, score = 0.6224999999999999
np.mean(results), mse, len(results) =  0.6330619469026548 6961.9893 2260
slice = train, score = 0.6330619469026548
np.mean(results), mse, len(results) =  0.5395755182625864 7021.2437 1013
slice = test, score = 0.5395755182625864

=== Iteration 80000 ===
np.mean(results), mse, len(results) =  0.6201 7042.486 100
slice = key, score = 0.6201
np.mean(results), mse, len(results) =  0.630066371681416 7048.9565 2260
slice = train, score = 0.630066371681416
np.mean(results), mse, len(results) =  0.5388647581441263 7111.7026 1013
slice = test, score = 0.5388647581441263

=== Iteration 81000 ===
np.mean(results), mse, len(results) =  0.623 7102.3633 100
slice = key, score = 0.623
np.mean(results), mse, len(results) =  0.6346238938053097 7076.3843 2260
slice = train, score = 0.6346238938053097
np.mean(results), mse, len(results) =  0.5408094768015795 7143.601 1013
slice = test, score = 0.5408094768015795

=== Iteration 82000 ===
np.mean(results), mse, len(results) =  0.6217 7213.7275 100
slice = key, score = 0.6217
np.mean(results), mse, len(results) =  0.6334292035398231 7207.454 2260
slice = train, score = 0.6334292035398231
np.mean(results), mse, len(results) =  0.5421125370187562 7270.582 1013
slice = test, score = 0.5421125370187562

=== Iteration 83000 ===
np.mean(results), mse, len(results) =  0.6226000000000002 7312.856 100
slice = key, score = 0.6226000000000002
np.mean(results), mse, len(results) =  0.6348185840707965 7289.0566 2260
slice = train, score = 0.6348185840707965
np.mean(results), mse, len(results) =  0.5405824284304047 7350.2876 1013
slice = test, score = 0.5405824284304047

=== Iteration 84000 ===
np.mean(results), mse, len(results) =  0.6258 7412.0234 100
slice = key, score = 0.6258
np.mean(results), mse, len(results) =  0.6361548672566372 7381.1855 2260
slice = train, score = 0.6361548672566372
np.mean(results), mse, len(results) =  0.5399506416584403 7446.4883 1013
slice = test, score = 0.5399506416584403

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.6272000000000001 7441.842 100
slice = key, score = 0.6272000000000001
np.mean(results), mse, len(results) =  0.6361592920353981 7459.562 2260
slice = train, score = 0.6361592920353981
np.mean(results), mse, len(results) =  0.5388252714708787 7536.038 1013
slice = test, score = 0.5388252714708787

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6236999999999999 7541.564 100
slice = key, score = 0.6236999999999999
np.mean(results), mse, len(results) =  0.6366017699115044 7530.1675 2260
slice = train, score = 0.6366017699115044
np.mean(results), mse, len(results) =  0.5398815399802567 7602.809 1013
slice = test, score = 0.5398815399802567

=== Iteration 87000 ===
np.mean(results), mse, len(results) =  0.6254 7622.9546 100
slice = key, score = 0.6254
np.mean(results), mse, len(results) =  0.637353982300885 7625.302 2260
slice = train, score = 0.637353982300885
np.mean(results), mse, len(results) =  0.5409476801579467 7695.1816 1013
slice = test, score = 0.5409476801579467

=== Iteration 88000 ===
np.mean(results), mse, len(results) =  0.6239999999999999 7644.096 100
slice = key, score = 0.6239999999999999
np.mean(results), mse, len(results) =  0.6342035398230089 7663.6294 2260
slice = train, score = 0.6342035398230089
np.mean(results), mse, len(results) =  0.5387364264560711 7748.1226 1013
slice = test, score = 0.5387364264560711

=== Iteration 89000 ===
np.mean(results), mse, len(results) =  0.6244999999999999 7671.2817 100
slice = key, score = 0.6244999999999999
np.mean(results), mse, len(results) =  0.635349557522124 7709.1396 2260
slice = train, score = 0.635349557522124
np.mean(results), mse, len(results) =  0.5388351431391906 7795.646 1013
slice = test, score = 0.5388351431391906

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.6245 7821.1206 100
slice = key, score = 0.6245
np.mean(results), mse, len(results) =  0.6356017699115044 7848.8086 2260
slice = train, score = 0.6356017699115044
np.mean(results), mse, len(results) =  0.5396051332675222 7937.391 1013
slice = test, score = 0.5396051332675222

=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.6260999999999999 7930.5195 100
slice = key, score = 0.6260999999999999
np.mean(results), mse, len(results) =  0.6374203539823009 7918.435 2260
slice = train, score = 0.6374203539823009
np.mean(results), mse, len(results) =  0.5413228035538007 7997.2466 1013
slice = test, score = 0.5413228035538007

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.6271000000000001 7915.5513 100
slice = key, score = 0.6271000000000001

np.mean(results), mse, len(results) =  0.6372522123893806 7945.881 2260
slice = train, score = 0.6372522123893806
np.mean(results), mse, len(results) =  0.5417176702862784 8021.7773 1013
slice = test, score = 0.5417176702862784

=== Iteration 93000 ===
np.mean(results), mse, len(results) =  0.6283 8033.187 100
slice = key, score = 0.6283
np.mean(results), mse, len(results) =  0.6394557522123894 8039.9277 2260
slice = train, score = 0.6394557522123894
np.mean(results), mse, len(results) =  0.5431095755182627 8117.227 1013
slice = test, score = 0.5431095755182627

=== Iteration 94000 ===
np.mean(results), mse, len(results) =  0.6260000000000001 8176.9834 100
slice = key, score = 0.6260000000000001
np.mean(results), mse, len(results) =  0.6368362831858407 8192.935 2260
slice = train, score = 0.6368362831858407
np.mean(results), mse, len(results) =  0.53928923988154 8285.141 1013
slice = test, score = 0.53928923988154

=== Iteration 95000 ===
np.mean(results), mse, len(results) =  0.6278 8112.354 100
slice = key, score = 0.6278
np.mean(results), mse, len(results) =  0.6375796460176991 8136.3438 2260
slice = train, score = 0.6375796460176991
np.mean(results), mse, len(results) =  0.541964461994077 8216.194 1013
slice = test, score = 0.541964461994077

=== Iteration 96000 ===
np.mean(results), mse, len(results) =  0.6235 8261.566 100
slice = key, score = 0.6235
np.mean(results), mse, len(results) =  0.635504424778761 8290.455 2260
slice = train, score = 0.635504424778761
np.mean(results), mse, len(results) =  0.5398617966436329 8373.116 1013
slice = test, score = 0.5398617966436329

=== Iteration 97000 ===
np.mean(results), mse, len(results) =  0.6271 8321.99 100
slice = key, score = 0.6271
np.mean(results), mse, len(results) =  0.6379336283185841 8366.377 2260
slice = train, score = 0.6379336283185841
np.mean(results), mse, len(results) =  0.5401974333662389 8455.101 1013
slice = test, score = 0.5401974333662389

=== Iteration 98000 ===
np.mean(results), mse, len(results) =  0.6281 8402.016 100
slice = key, score = 0.6281
np.mean(results), mse, len(results) =  0.6353628318584071 8418.119 2260
slice = train, score = 0.6353628318584071
np.mean(results), mse, len(results) =  0.5395162882527147 8492.778 1013
slice = test, score = 0.5395162882527147

=== Iteration 99000 ===
np.mean(results), mse, len(results) =  0.6304000000000001 8456.527 100
slice = key, score = 0.6304000000000001
np.mean(results), mse, len(results) =  0.6369292035398231 8497.843 2260
slice = train, score = 0.6369292035398231
np.mean(results), mse, len(results) =  0.5382033563672262 8580.131 1013
slice = test, score = 0.5382033563672262
last loss =  tf.Tensor(-0.0044855885, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6369646017699115 8588.509 2260
np.mean(results), mse, len(results) =  0.5387068114511353 8685.359 1013
"""

In [76]:
def get_train_test(s):
    v = s.strip().split("\n")
    try:
        train = [v_i for v_i in v if v_i.startswith("slice = train")][0]
        test = [v_i for v_i in v if v_i.startswith("slice = test")][0]
        return float(train.split(" = ")[-1]), float(test.split(" = ")[-1])
    except Exception as e:
        print("wtf")
        return (-1, -1)
    
def get_best(s):
    return sorted([get_train_test(x) for x in s.strip().split("=== Iteration") if x], reverse=True)[:3]

In [77]:
get_best(s)

[(0.6394557522123894, 0.5431095755182627),
 (0.6379336283185841, 0.5401974333662389),
 (0.6375796460176991, 0.541964461994077)]

In [ ]:
# ApproxNDCG
# ApproxMAP ???
# No W, biases, etc
# Moar Layers
# Approx NDCG + AnnCur init
# WarmUP-training + aftertraining
# Best Clustering + our
# Anncur || trainable -> emb

### games

In [12]:
L = 7000
N = 1000
DA = 50

In [33]:
!ls -lah

total 3.9G
drwxr-xr-x 10 shevkunov shevkunov 4.0K Sep  6 11:48 .
drwxr-xr-x 56 shevkunov shevkunov 4.0K May  6 01:49 ..
drwxrwxr-x  2 shevkunov shevkunov 4.0K Aug 20 15:53 doctor_who
-rwxr-xr-x  1 shevkunov shevkunov 108M Apr 14 15:02 games-all.json
drwxr-xr-x  2 shevkunov shevkunov 4.0K Aug 24 19:45 .ipynb_checkpoints
-rw-rw-r--  1 shevkunov shevkunov 1.3G May 19 12:26 log.local.bin
-rwxr-xr-x  1 shevkunov shevkunov 691M May 19 11:32 log.local.bin-old
-rw-rw-r--  1 shevkunov shevkunov 1.8G Jul 22 00:21 log.local.logtime2.bin
-rw-r--r--  1 shevkunov shevkunov 5.5K Jul  9 17:54 matrix_approx_zeshel.py
drwxrwxr-x  2 shevkunov shevkunov 4.0K Sep  2 17:08 military
-rwxr-xr-x  1 shevkunov shevkunov 345K Feb  8  2023 proof-of-concept.ipynb
-rw-rw-r--  1 shevkunov shevkunov 438K Jul 12 12:23 proof-of-concept-open-data-round1.ipynb
-rw-rw-r--  1 shevkunov shevkunov 898K Jul 30 16:51 proof-of-concept-open-data-round2-anncurcomparison.ipynb
-rw-rw-r--  1 shevkunov shevkunov  60K Aug 24 13:30 pro

In [13]:
ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA) # .set_top_games_as_key()
ev([
    AnnCUR(ctx)
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034



model =  AnnCur(139929851077648)
np.mean(results), mse, len(results) =  0.6764900947459087 1.9841502796717772 4644
np.mean(results), mse, len(results) =  0.6652409046214356 2.8454261374203558 2034
0.6764900947459087 0.6652409046214356


In [16]:
ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA).set_top_games_as_key()

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034


In [27]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7fd33860bb20>
np.mean(results), mse, len(results) =  0.5083527131782946 2.1090857474029794 4644
np.mean(results), mse, len(results) =  0.5076548672566372 2.214983388749716 2034
0.5083527131782946 0.5076548672566372



model =  AnnCur(140545160627632)
np.mean(results), mse, len(results) =  0.7693260120585702 0.5653017362404418 4644
np.mean(results), mse, len(results) =  0.762251720747296 0.5927785494345469 2034
0.7693260120585702 0.762251720747296



model =  AnnCur(key_games=np.arange(100) - 140531535266816)
np.mean(results), mse, len(results) =  0.6579565030146425 1.9767935375740953 4644
np.mean(results), mse, len(results) =  0.6489528023598821 6.906013812566882 2034
0.6579565030146425 0.6489528023598821



model =  AnnCur(random - 140531535278288)
np.mean(results), mse, len(results) =  0.6810099052540913 1.9702658686057843 4644
np.mean(results), mse, len(results) =  0.6697246804326451 2.180510915221889 2034
0.6810099052540913 0.66972468043264

In [34]:
!ls stand

log.local.logtime2.txt.old  log.local.txt  log.local.txt.tar.gz


In [13]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.609521994482665e-120
2.609521994482665e-120
101 4769 2088


In [14]:
ev([
    Popular(ctx),
    AnnCUR(ctx)
])




model =  <__main__.Popular object at 0x7ff62130f6d0>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(140695785624080)
np.mean(results), mse, len(results) =  0.5976535961417488 0.17482987219177043 4769
np.mean(results), mse, len(results) =  0.5860488505747127 0.19354859381230788 2088
0.5976535961417488 0.5860488505747127


In [15]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA).set_top_games_as_key()

/tmp/ipykernel_690140/664571995.py:33: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i, line in tqdm.tqdm_notebook(enumerate(f)):


0it [00:00, ?it/s]

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.609521994482665e-120
2.609521994482665e-120
101 4769 2088


In [16]:
ev([
    Popular(ctx),
    AnnCUR(ctx)
])




model =  <__main__.Popular object at 0x7f3894f410c0>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(139872009046960)
np.mean(results), mse, len(results) =  0.6778381211994128 0.24971886538105065 4769
np.mean(results), mse, len(results) =  0.669477969348659 0.27141239614050905 2088
0.6778381211994128 0.669477969348659


In [17]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7f3677c5d450>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(139880993919168)
np.mean(results), mse, len(results) =  0.6778381211994128 0.24971886538105065 4769
np.mean(results), mse, len(results) =  0.669477969348659 0.27141239614050905 2088
0.6778381211994128 0.669477969348659



model =  AnnCur(key_games=np.arange(100) - 139886535450944)
np.mean(results), mse, len(results) =  0.568666387083246 0.2101395930999089 4769
np.mean(results), mse, len(results) =  0.5609003831417625 0.2306177104605779 2088
0.568666387083246 0.5609003831417625



model =  AnnCur(random - 139886535448400)
np.mean(results), mse, len(results) =  0.5932228978821555 0.1733168362865605 4769
np.mean(results), mse, len(results) =  0.584176245210728 0.1904597187905242 2088
0.5932228978821555 0.58417624521072

In [13]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

ctx.set_kmeans_games_as_key()

N = 20000

m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': False,
    'verbose': False, 'loss': 'softmax',
    'loss_batch': 128, 'loss_q': 0.99,
    'n': N,
    # 'ubatch': 512
    'score_verbose': 1000,
    'trainable_items': True  # <<< DIFF HERE
})
m.fit()
m.get_score("train"), m.get_score("test")


del ctx
gc.collect()

del m
gc.collect()

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.6095219944834066e-120
2.6095219944834066e-120
101 4769 2088


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


self.embed_users['train'].shape =  (4769, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (4769, 101)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.26039603960396046 7212.415762310138 101
slice = key, score = 0.26039603960396046
np.mean(results), mse, len(results) =  0.24992451247641018 6721.820244320349 4769
slice = train, score = 0.24992451247641018
np.mean(results), mse, len(results) =  0.2504501915708813 6756.845014144603 2088
slice = test, score = 0.2504501915708813

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5113861386138614 25556.81713193163 101
slice = key, score = 0.5113861386138614
np.mean(results), mse, len(results) =  0.5061438456699519 27426.627092275852 4769
slice = train, score = 0.5061438456699519
np.mean(results), mse, len(results) =  0.503227969348659 26833.93909681337 2088
slice = test, scor

14321

In [22]:
!ls stand -lah

total 18G
drwxrwxr-x 10 shevkunov dpt_yandex_monetize_banner_adv_relevance_dep24304_dep87392 4.0K Jun  6 17:12 .
drwxrwxr-x 10 shevkunov dpt_yandex_monetize_banner_adv_relevance_dep24304_dep87392 4.0K Sep 10 12:20 ..
drwxrwxr-x  3 shevkunov dpt_yandex_monetize_banner_adv_relevance_dep24304_dep87392 4.0K Feb  7  2023 attempt
drwxrwxr-x  3 shevkunov dpt_yandex_monetize_banner_adv_relevance_dep24304_dep87392 4.0K Feb  7  2023 bin
drwxrwxr-x  3 shevkunov dpt_yandex_monetize_banner_adv_relevance_dep24304_dep87392 4.0K Jun  6 00:05 config
-rw-rw-r--  1 shevkunov dpt_yandex_monetize_banner_adv_relevance_dep24304_dep87392  16K Feb 20  2023 daemon.json
drwxrwxr-x  3 shevkunov dpt_yandex_monetize_banner_adv_relevance_dep24304_dep87392 4.0K Feb  7  2023 dssm
-rw-r--r--  1 root      root                                                          0 Jun  6 15:43 dumped_requests.gz
-rw-r--r--  1 root      root                                                        21K Jun  6 15:43 dumped_requests.gz.PR

In [21]:
ctx = load(L, raw_path = "stand/log.local.txt.old", path="log.local.txt.old.bin", seed=17, det_attempts=DA)

ctx.set_kmeans_games_as_key()

N = 20000

m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': False,
    'verbose': False, 'loss': 'softmax',
    'loss_batch': 128, 'loss_q': 0.99,
    'n': N,
    # 'ubatch': 512
    'score_verbose': 1000,
    'trainable_items': True  # <<< DIFF HERE
})
m.fit()
m.get_score("train"), m.get_score("test")


del ctx
gc.collect()

del m
gc.collect()

/tmp/ipykernel_593987/307185960.py:33: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i, line in tqdm.tqdm_notebook(enumerate(f)):


0it [00:00, ?it/s]

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


self.embed_users['train'].shape =  (4644, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (4644, 101)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.36346534653465357 94.12616437504039 101
slice = key, score = 0.36346534653465357
np.mean(results), mse, len(results) =  0.359394918173988 52.60288033215936 4644
slice = train, score = 0.359394918173988
np.mean(results), mse, len(results) =  0.3786627335299902 54.505824375923616 2034
slice = test, score = 0.3786627335299902

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5869306930693069 4029.8515224223434 101
slice = key, score = 0.5869306930693069
np.mean(results), mse, len(results) =  0.6523664944013782 1669.4905893345801 4644
slice = train, score = 0.6523664944013782
np.mean(results), mse, len(results) =  0.6319469026548673 1742.543577882201 2034
slice = test, score

np.mean(results), mse, len(results) =  0.6604719764011799 183089.11505654827 2034


13963